# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [226]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text8 = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text8)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [227]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
# - Then use nltk.sent_tokenize.
#
# Return: sentences (list of strings)

# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)
# TODO: apply sent_tokenize

# print(sentences)
def protect_acronym_dots(text):
    # Use regex to find sequences of uppercase letters separated by dots and replaces those dots with a placeholder.
    # (?<=[A-Z]) matches a position after an uppercase letter.
    # \. matches the dot.
    # (?=[A-Z]) matches a position before an uppercase letter.
    # This avoids matching the final dot of a sentence if its before a lowercase word.
    return re.sub(r'(?<=[A-Z])\.(?=[A-Z])', '[[DOT]]', text)
def restore_acronym_dots(text):
    # Reverts the temporary placeholder back to dots.
    return text.replace('[[DOT]]', '.')
# Protect acronyms
protected_text = protect_acronym_dots(text8)
# Apply sent_tokenize from NLTK to the new text to split it into different sentences
raw_sentences = sent_tokenize(protected_text)
# Restore the dots in all the different sentences 
sentences = [restore_acronym_dots(s) for s in raw_sentences]
print(f"Number of sentences: {len(sentences)}")
for i, s in enumerate(sentences):
    print(f"Sentence {i+1}: {s}")

Number of sentences: 3
Sentence 1: In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.
Sentence 2: He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O.
Sentence 3: A report valued the project at $3.2 billion.


## Q2

In [228]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
#
# Return: text_norm
# print(text_norm)
def normalize_text(input_text):
    # Remove dots in acronyms 
    # This works for capital letters before the dots and to maintain the letter.
    text = re.sub(r'(?<=[A-Z])\.(?=[A-Z.]| )', '', input_text)
    # High in meters to centimeters
    # Regex captures the float value before the 'm'.
    def convert_height(match):
        meters = float(match.group(1))
        cms = int(round(meters * 100))
        return f"{cms} centimeters"
    text = re.sub(r'(\d+\.\d+)m', convert_height, text)
    # The billion in numbers to the words of the number 
    # Map digits 0-9 to their equivalents in words.
    num_to_word = {
        '0': 'zero', '1': 'one', '2': 'two', '3': 'three', '4': 'four',
        '5': 'five', '6': 'six', '7': 'seven', '8': 'eight', '9': 'nine'
    }
    def convert_money(match):
        digit1 = match.group(1)
        digit2 = match.group(2)
        return f"{num_to_word[digit1]} point {num_to_word[digit2]} billion"
    # Use Regex to search for $, a digit, a dot, a digit, and the word 'billion' and replace it
    text = re.sub(r'\$(\d)\.(\d)\s+billion', convert_money, text)
    return text
# we apply the normalization function the original text
text_norm = normalize_text(text8)
print(text_norm)

In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q3

In [229]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case

# print(text_case)
def process_casing(text):
    # Join  a specific multi-word 
    text = text.replace("Sam Altman", "Sam_Altman")
    # Split text into words and process each one according to the rules
    words = text.split()
    processed_words = []
    for word in words:
        # Strip trailing punctuation for the check, but keep the original one for output
        clean_word = word.rstrip('.,')
        # Check if it's an acronym, mixed case, or a joined entity
        is_acronym = clean_word.isupper() and len(clean_word) > 1
        is_mixed_case = any(c.isupper() for c in clean_word[1:])
        is_joined_entity = "_" in clean_word
        if is_acronym or is_mixed_case or is_joined_entity:
            processed_words.append(word)
        else:
            processed_words.append(word.lower())
    return " ".join(processed_words)
# We use the original test
text_case = process_casing(text8)
print(text_case)

in mid-February 2026, the CEO of OpenAI, Sam_Altman, visited barcelona. he is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. a report valued the project at $3.2 billion.


## Q4

In [230]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

tokens = None

# print(tokens)
# We use the original text
# word_tokenize from nltk uses the Punkt model 
# splitting as it handles contractions and punctuation rules of English.
tokens = word_tokenize(text8)
#  Words like "Barcelona." becomes ["Barcelona", "."] instead of one single unit.
print(tokens)


['In', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam', 'Altman', ',', 'visited', 'Barcelona', '.', 'He', 'is', '1.86m', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'U.P.C', '.', 'and', 'U.N.E.S.C.O', '.', 'A', 'report', 'valued', 'the', 'project', 'at', '$', '3.2', 'billion', '.']


## Q5

In [231]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop

tokens_nostop = None

# print(tokens_nostop)


## Q6

In [232]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

bigrams = None

# print(bigrams)


## Q7

In [233]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = None
context_counts = None
model = None

def predict_next(prev_word, model, top_k=3):
    # TODO
    return None

# Example:
# print(predict_next("OpenAI", model, top_k=3))


## Q8

In [234]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [235]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = None
FP = None
FN = None
TN = None

accuracy = None
precision = None
recall = None
f1 = None

# print(accuracy, precision, recall, f1)
